# 02. Step 1: Aspect & Opinion Co-Extraction (BERT-CRF & Implicit Detection)

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook trains and evaluates **Step 1** of the ACOS framework:
- **Model Architecture (`BertForQuadABSA`):** Pretrained BERT backbone coupled with a Linear projection layer and a Conditional Random Field (CRF) sequence tagger for extracting explicit aspect and opinion spans (`B-A`, `I-A`, `B-O`, `I-O`, `O`), plus auxiliary multi-label classification heads on the `[CLS]` token for detecting implicit aspects and opinions (`[-1, -1]`).
- **Model Checkpointing:** Saves the best fine-tuned model weights (`pytorch_model.bin`), configuration (`config.json`), and vocabulary (`vocab.txt`) to `checkpoints/step1_best/`.
- **Output Artifacts:** Produces `pred4pipeline.txt` (input bridge for Step 2), training loss/F1 curves (`plots/`), and structured CSV metrics tables (`csv/`).

## 1. Environment & Module Imports

In [ ]:
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3
import os
import sys
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm, trange

# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from modeling import BertForQuadABSA
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes, convert_examples_to_features
from eval_metrics import pred_eval

# 3. Import colab_utils with fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Active PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 2. Configuration & Hyperparameters
Select domain (`rest16` or `laptop`), batch size, epochs, and initialize the timestamped output directory.

In [ ]:
# Pipeline Configuration
DOMAIN = "rest16"              # 'rest16' (Restaurant-ACOS) or 'laptop' (Laptop-ACOS)
TASK_NAME = "quad"
MODEL_TYPE = "quad"
DO_TRAIN = True                # Set to False to skip training and evaluate saved checkpoint
DO_EVAL = True
MAX_SEQ_LENGTH = 128
TRAIN_BATCH_SIZE = 24
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 15          # Default is 30, 15 is great for fast Colab training
WARMUP_PROPORTION = 0.1
SEED = 42

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Pretrained BERT Directory
bert_model_dir = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_model_dir)

# Data directory
data_dir = extract_dir

# Initialize timestamped output session directory
results_base = os.path.join(base_project_dir, "results")
session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
step1_checkpoint_dir = session_dirs["step1_checkpoint"]

print(f"📁 Step 1 Checkpoint will be saved to: {step1_checkpoint_dir}")

## 3. Data Loading & Feature Conversion
Load train, dev, and test examples and convert them into PyTorch TensorDatasets.

In [ ]:
# Initialize Tokenizer and Processor
tokenizer = BertTokenizer.from_pretrained(bert_model_dir, do_lower_case=True)
processor = processors[TASK_NAME]()
label_list = processor.get_labels(DOMAIN)
num_labels = len(label_list[1])
label_map_seq = {label: i for i, label in enumerate(label_list[1])}

print(f"Sequence Labels ({num_labels}): {label_list[1]}")

# Convert Dev/Eval Data
eval_examples = processor.get_dev_examples(data_dir, DOMAIN)
eval_features = convert_examples_to_features(eval_examples, label_list, MAX_SEQ_LENGTH, tokenizer, output_modes[TASK_NAME], TASK_NAME, domain_type=DOMAIN)

all_input_ids = torch.tensor([f.aspect_input_ids for f in eval_features], dtype=torch.long)
all_input_mask = torch.tensor([f.aspect_input_mask for f in eval_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.aspect_segment_ids for f in eval_features], dtype=torch.long)
all_label_ids = torch.tensor([f.aspect_ids for f in eval_features], dtype=torch.long)
all_exist_imp_aspect = torch.tensor([f.exist_imp_aspect for f in eval_features], dtype=torch.long)
all_exist_imp_opinion = torch.tensor([f.exist_imp_opinion for f in eval_features], dtype=torch.long)
all_tokens_len = torch.tensor([f.tokens_len for f in eval_features], dtype=torch.long)

eval_data = TensorDataset(all_tokens_len, all_input_ids, all_input_mask, all_label_ids, all_segment_ids, all_exist_imp_aspect, all_exist_imp_opinion)
eval_sampler = SequentialSampler(eval_data)
eval_dataloader = DataLoader(eval_data, sampler=eval_sampler, batch_size=EVAL_BATCH_SIZE)

# Load Gold Test Data
eval_gold = []
test_quad_file = os.path.join(data_dir, "tokenized_data", f"{DOMAIN}_test_quad_bert.tsv")
with open(test_quad_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip().split("\t")
        cur_text = tokenizer.convert_tokens_to_ids(line[0].split(" "))
        cur_exist_imp_aspect = 0
        cur_exist_imp_opinion = 0
        aspect_labels = [label_map_seq['O'] for _ in range(MAX_SEQ_LENGTH)]
        for quad in line[1:]:
            cur_aspect = quad.split(' ')[0]
            cur_opinion = quad.split(' ')[-1]
            a_st, a_ed = int(cur_aspect.split(',')[0]), int(cur_aspect.split(',')[1])
            if a_ed != -1:
                aspect_labels[a_st] = label_map_seq['B-A']
                for i in range(a_st+1, a_ed):
                    aspect_labels[i] = label_map_seq['I-A']
            else:
                cur_exist_imp_aspect = 1
            o_st, o_ed = int(cur_opinion.split(',')[0]), int(cur_opinion.split(',')[1])
            if o_ed != -1:
                aspect_labels[o_st] = label_map_seq['B-O']
                for i in range(o_st+1, o_ed):
                    aspect_labels[i] = label_map_seq['I-O']
            else:
                cur_exist_imp_opinion = 1
        eval_gold.append([cur_text, [aspect_labels, cur_exist_imp_aspect, cur_exist_imp_opinion]])

input_text = [ele[0] for ele in eval_gold]
pairgold = [item for ele in eval_gold for item in ele[1]]
eval_gold = [input_text, pairgold]
print(f"✅ Evaluated Gold Test Samples Loaded: {len(input_text)}")

## 4. Model Initialization: `BertForQuadABSA`
Instantiate BERT + Linear + CRF sequence tagger model.

In [ ]:
model = BertForQuadABSA.from_pretrained(bert_model_dir, num_labels=num_labels)
model.to(device)
print(f"✅ Initialized BertForQuadABSA model ({sum(p.numel() for p in model.parameters()):,} parameters).")

## 5. Training Loop with Model Checkpoint Persistence
Performs fine-tuning with Adam optimizer and linear learning rate warmup. Evaluates validation Micro-F1 after every epoch and saves the best model checkpoint to `checkpoints/step1_best/`.

In [ ]:
if DO_TRAIN:
    train_examples = processor.get_train_examples(data_dir, DOMAIN)
    train_features = convert_examples_to_features(train_examples, label_list, MAX_SEQ_LENGTH, tokenizer, output_modes[TASK_NAME], TASK_NAME, domain_type=DOMAIN)
    
    tr_input_ids = torch.tensor([f.aspect_input_ids for f in train_features], dtype=torch.long)
    tr_input_mask = torch.tensor([f.aspect_input_mask for f in train_features], dtype=torch.long)
    tr_segment_ids = torch.tensor([f.aspect_segment_ids for f in train_features], dtype=torch.long)
    tr_label_ids = torch.tensor([f.aspect_ids for f in train_features], dtype=torch.long)
    tr_exist_imp_aspect = torch.tensor([f.exist_imp_aspect for f in train_features], dtype=torch.long)
    tr_exist_imp_opinion = torch.tensor([f.exist_imp_opinion for f in train_features], dtype=torch.long)
    tr_tokens_len = torch.tensor([f.tokens_len for f in train_features], dtype=torch.long)
    
    train_data = TensorDataset(tr_tokens_len, tr_input_ids, tr_input_mask, tr_label_ids, tr_segment_ids, tr_exist_imp_aspect, tr_exist_imp_opinion)
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=TRAIN_BATCH_SIZE)
    
    num_train_optimization_steps = len(train_dataloader) * NUM_TRAIN_EPOCHS
    
    # Optimizer setup
    param_optimizer = list(model.named_parameters())
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = BertAdam(optimizer_grouped_parameters, lr=LEARNING_RATE, warmup=WARMUP_PROPORTION, t_total=num_train_optimization_steps)
    
    print(f"🚀 Starting Step 1 Training: {NUM_TRAIN_EPOCHS} Epochs, {len(train_dataloader)} Steps/Epoch...")
    
    # Mock args for pred_eval helper
    class ArgsHelper:
        def __init__(self):
            self.output_dir = session_dirs["logs"]
            self.max_seq_length = MAX_SEQ_LENGTH
    eval_args = ArgsHelper()
    
    import logging
    logger = logging.getLogger("Step1")
    
    best_val_f1 = 0.0
    training_history = []
    
    for epoch in range(1, NUM_TRAIN_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch}/{NUM_TRAIN_EPOCHS}")):
            batch = tuple(t.to(device) for t in batch)
            _len, _ids, _mask, _labels, _seg_ids, _imp_a, _imp_o = batch
            
            loss, _ = model(
                aspect_input_ids=_ids,
                aspect_labels=_labels,
                aspect_token_type_ids=_seg_ids,
                aspect_attention_mask=_mask,
                exist_imp_aspect=_imp_a,
                exist_imp_opinion=_imp_o
            )
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_dataloader)
        
        # Evaluation
        model.eval()
        val_res = pred_eval(epoch, eval_args, logger, tokenizer, model, eval_dataloader, eval_gold, label_list, device, TASK_NAME, eval_type='test')
        
        val_p = val_res.get('precision', 0.0)
        val_r = val_res.get('recall', 0.0)
        val_f1 = val_res.get('micro-F1', 0.0)
        
        print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f} | Test P: {val_p*100:.2f}% | R: {val_r*100:.2f}% | Micro-F1: {val_f1*100:.2f}%")
        
        training_history.append({
            "epoch": epoch,
            "loss": avg_loss,
            "precision": val_p,
            "recall": val_r,
            "micro-F1": val_f1
        })
        
        # SAVE BEST CHECKPOINT
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            print(f"🔥 New Best Test F1: {best_val_f1*100:.2f}%! Saving model checkpoint to {step1_checkpoint_dir}...")
            
            # Save weights
            torch.save(model.state_dict(), os.path.join(step1_checkpoint_dir, "pytorch_model.bin"))
            model.config.to_json_file(os.path.join(step1_checkpoint_dir, "config.json"))
            tokenizer.save_vocabulary(step1_checkpoint_dir)
            
            # Save metadata
            metadata = {
                "epoch": epoch,
                "best_micro_f1": best_val_f1,
                "precision": val_p,
                "recall": val_r,
                "domain": DOMAIN,
                "task": "Step1_Aspect_Opinion_Extraction"
            }
            with open(os.path.join(step1_checkpoint_dir, "checkpoint_metadata.json"), "w") as mf:
                json.dump(metadata, mf, indent=2)
                
    # Export Training Curves & CSV
    plot_history_path = os.path.join(session_dirs["plots"], "03_step1_training_loss_f1_curve.png")
    csv_history_path = os.path.join(session_dirs["csv"], "step1_training_history.csv")
    plot_training_history(training_history, task_name="Step 1 (BERT-CRF)", output_plot_path=plot_history_path, output_csv_path=csv_history_path)
    print(f"💾 Saved Step 1 Training History CSV: {csv_history_path}")

## 6. Standalone Checkpoint Loading & Final Test Set Inference
Loads the fine-tuned model checkpoint from `checkpoints/step1_best/` and generates the pipeline prediction file `logs/pred4pipeline.txt`.

In [ ]:
print(f"📥 Loading best fine-tuned Step 1 checkpoint from: {step1_checkpoint_dir}")
model = BertForQuadABSA.from_pretrained(step1_checkpoint_dir, num_labels=num_labels)
model.to(device)
model.eval()

class ArgsHelper:
    def __init__(self):
        self.output_dir = session_dirs["logs"]
        self.max_seq_length = MAX_SEQ_LENGTH
eval_args = ArgsHelper()

import logging
logger = logging.getLogger("Step1_Final")
final_res = pred_eval('best_checkpoint', eval_args, logger, tokenizer, model, eval_dataloader, eval_gold, label_list, device, TASK_NAME, eval_type='test')

print("\n🏆 Final Step 1 Test Results:")
for k, v in final_res.items():
    print(f"  - {k}: {v*100:.2f}%")

# Verify pred4pipeline.txt was generated
pred_file = os.path.join(session_dirs["logs"], "pred4pipeline.txt")
if os.path.exists(pred_file):
    with open(pred_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"\n✅ 'pred4pipeline.txt' generated ({len(lines)} predicted sentences). Ready for Step 3!")
    print("Preview first 3 predicted lines:")
    for l in lines[:3]:
        print(f"  {l.strip()}")

## 7. Display Step 1 Training Loss & Metrics Curve

In [ ]:
from IPython.display import Image, display
plot_path = os.path.join(session_dirs["plots"], "03_step1_training_loss_f1_curve.png")
if os.path.exists(plot_path):
    display(Image(plot_path))

print("✨ Step 1 finished! Proceed to '03_ACOS_Step1_to_Step2_Pair_Generation.ipynb'!")